In [ ]:
# ==========================================================
# EVALUATION COMPLETE (MULTICLASSE + BINAIRE)
# ==========================================================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    classification_report,
    balanced_accuracy_score,
    roc_curve,
    auc
)

# ==========================================================
# Mask-Guided Variable Window Pooling (PyTorch version)
#   + expose last_hk and last_m for regularization
# ==========================================================

class MaskGuidedVariablePooling(nn.Module):
    def __init__(self, K, tau=1):
        super().__init__()
        self.K = K
        self.tau = tau

        self.h_mlp = nn.Sequential(
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

        # Exposed values for regularization
        self.last_hk = None   # [B]
        self.last_m  = None   # [B, W]

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        device = x.device

        # 1) Predict temporal horizon
        pooled = x.mean(dim=[2,3])          # [B, C]
        h_k = self.h_mlp(pooled).squeeze(1) # [B]

        # 2) Temporal positions
        t = torch.linspace(0.0, 1.0, W, device=device).unsqueeze(0)  # [1,W]

        # 3) Temporal mask
        m = torch.sigmoid(self.tau * (h_k.unsqueeze(1) - t))          # [B,W]

        # Save for regularization
        self.last_hk = h_k
        self.last_m  = m

        # 4) Slope
        s = torch.relu(m[:, :-1] - m[:, 1:])  # [B,W-1]

        eps = 1e-8
        p = (s + eps) / (s.sum(dim=1, keepdim=True) + eps)  # [B,W-1]
        cdf = torch.cumsum(p, dim=1)                         # [B,W-1]
        cdf[:, -1] = 1.0

        # 5) Quantile cuts
        quantiles = torch.linspace(0, 1, self.K+1, device=device)[1:-1]  # [K-1]

        cuts = []
        for q in quantiles:
            cut = torch.argmax((cdf >= q).int(), dim=1)  # [B]
            cuts.append(cut)

        if len(cuts) > 0:
            cuts = torch.stack(cuts, dim=1)              # [B,K-1]
        else:
            cuts = torch.empty(B,0,dtype=torch.long,device=device)

        # 6) Pooling
        outputs = []
        prev = torch.zeros(B, dtype=torch.long, device=device)
        idx = torch.arange(W, device=device).unsqueeze(0)  # [1,W]

        for r in range(self.K):
            if r < self.K-1:
                curr = cuts[:, r]
            else:
                curr = torch.full((B,), W-1, device=device, dtype=torch.long)

            seg_mask = (idx >= prev.unsqueeze(1)) & (idx <= curr.unsqueeze(1))
            seg_mask = seg_mask.float()

            weights = seg_mask * m
            weights = weights / (weights.sum(dim=1, keepdim=True) + eps)

            pooled_seg = (x * weights.unsqueeze(1).unsqueeze(2)).sum(dim=3)  # sum over W
            outputs.append(pooled_seg)

            prev = curr + 1

        output = torch.stack(outputs, dim=3)  # [B,C,H,K]
        return output


# ==========================================================
# MODEL
# ==========================================================

class Model(nn.Module):
    def __init__(self, n_classes=15):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, (1,3))
        self.pool1 = MaskGuidedVariablePooling(K=7)

        self.conv2 = nn.Conv2d(16, 16, (1,2))
        self.pool2 = MaskGuidedVariablePooling(K=5)

        self.flatten = nn.Flatten()

        dummy = torch.zeros(1,1,82,20)
        dummy = torch.relu(self.conv1(dummy))
        dummy = self.pool1(dummy)
        dummy = torch.relu(self.conv2(dummy))
        dummy = self.pool2(dummy)
        flatten_size = dummy.numel()

        self.fc = nn.Sequential(
            nn.Linear(flatten_size, 64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,n_classes)
        )

    def forward(self,x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x



# ==========================================================
# MAPPING ORIGINAL
# ==========================================================

original_labels = {
    0: "BENIGN",
    1: "Bot",
    2: "DDoS",
    3: "DoS GoldenEye",
    4: "DoS Hulk",
    5: "DoS Slowhttptest",
    6: "DoS slowloris",
    7: "FTP-Patator",
    8: "Heartbleed",
    9: "Infiltration",
    10: "PortScan",
    11: "SSH-Patator",
    12: "Web Attack – Brute Force",
    13: "Web Attack – Sql Injection",
    14: "Web Attack – XSS"
}

# ==========================================================
# REGROUPEMENT
# ==========================================================

def regroup_classes(label_id):
    if label_id in [3, 4, 5, 6]:
        return "DoS"
    elif label_id in [12, 13, 14]:
        return "WebAttack"
    elif label_id in [7, 11]:
        return "Patator"
    else:
        return original_labels[label_id]


def evaluate_model(model, X, Y):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
    Y_tensor = torch.tensor(Y, dtype=torch.long)

    dataset = TensorDataset(X_tensor, Y_tensor)
    loader = DataLoader(dataset, batch_size=512)

    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in tqdm(loader, desc="Evaluation"):

            xb = xb.to(device)
            outputs = model(xb)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(yb.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    # ======================================================
    # MULTICLASSE ORIGINALE
    # ======================================================

    print("\n========== MULTICLASSE ORIGINALE ==========")

    print("Accuracy :", accuracy_score(all_labels, all_preds))
    print("F1 weighted:", f1_score(all_labels, all_preds, average="weighted"))
    print("Balanced accuracy:", balanced_accuracy_score(all_labels, all_preds))

    print("\nRapport détaillé :")
    print(classification_report(all_labels, all_preds, digits=4))

    # ======================================================
    # MULTICLASSE REGROUPEE
    # ======================================================

    print("\n========== MULTICLASSE REGROUPEE ==========")

    y_true_group = np.array([regroup_classes(l) for l in all_labels])
    y_pred_group = np.array([regroup_classes(l) for l in all_preds])

    print("Accuracy :", accuracy_score(y_true_group, y_pred_group))
    print("F1 weighted:", f1_score(y_true_group, y_pred_group, average="weighted"))
    print("Balanced accuracy:", balanced_accuracy_score(y_true_group, y_pred_group))

    print("\nRapport détaillé regroupé :")
    print(classification_report(y_true_group, y_pred_group, digits=4))

    # Matrice regroupée normalisée
    cm_group = confusion_matrix(y_true_group, y_pred_group)
    cm_group_norm = cm_group.astype(float) / cm_group.sum(axis=1, keepdims=True)
    cm_group_norm = np.nan_to_num(cm_group_norm)

    plt.figure(figsize=(10,8))
    disp = ConfusionMatrixDisplay(cm_group_norm)
    disp.plot(cmap="Blues", values_format=".2f")
    plt.title("Matrice regroupée (%) - Normalisée par ligne")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # ======================================================
    # BINAIRE
    # ======================================================

    print("\n========== BINAIRE ==========")

    y_true_bin = (all_labels != 0).astype(int)
    y_pred_bin = (all_preds != 0).astype(int)

    print("Accuracy :", accuracy_score(y_true_bin, y_pred_bin))
    print("F1 :", f1_score(y_true_bin, y_pred_bin))
    print("Balanced accuracy :", balanced_accuracy_score(y_true_bin, y_pred_bin))

    cm_bin = confusion_matrix(y_true_bin, y_pred_bin)
    cm_bin_norm = cm_bin.astype(float) / cm_bin.sum(axis=1, keepdims=True)
    cm_bin_norm = np.nan_to_num(cm_bin_norm)

    plt.figure(figsize=(5,5))
    disp = ConfusionMatrixDisplay(cm_bin_norm,
                                  display_labels=["Normal", "Attaque"])
    disp.plot(cmap="Reds", values_format=".2f")
    plt.title("Matrice Binaire (%)")
    plt.tight_layout()
    plt.show()

    # ROC
    attack_probs = all_probs[:,1:].sum(axis=1)
    fpr, tpr, _ = roc_curve(y_true_bin, attack_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve (Binaire)")
    plt.legend()
    plt.show()

# ==========================================================
# CHARGEMENT DONNEES TEST
# ==========================================================

print("--------------------Chargement données test--------------------")

X_input = np.load('./X_input_split_test/X_input_0.npy')

for i in range(1, 20):
    X_input = np.concatenate((
        X_input,
        np.load(f'./X_input_split_test/X_input_{i}.npy')
    ))

Y = np.load('./X_input_split_test/Y_test.npy')

print("Shape X:", X_input.shape)
print("Shape Y:", Y.shape)




# ==========================================================
# CHARGEMENT MODELE
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_eval = Model().to(device)
model_eval.load_state_dict(
    torch.load("./cicids_models/model_dynamic_cicids.pth",
               map_location=device)
)

print("\n--------------------Évaluation--------------------")
evaluate_model(model_eval, X_input, Y)
